# Learn a world model, then plan by imagination

**Track D · Scene & World Models** · maps to lesson D8 (world models).

A world model *learns the dynamics* of an environment so an agent can plan inside its imagination. Here: collect random experience in a 2D point-mass world, **train a dynamics model** `f(state, action) → next state`, then **plan** by rolling that model forward and picking the action sequence that reaches the goal (model-predictive control). Self-contained.

> CPU is fine.

In [ ]:
import os, torch, torch.nn as nn, matplotlib.pyplot as plt
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
STEPS = int(os.environ.get("STEPS", 1500))

## 1 · Environment — a 2D point mass
State = [x, y, vx, vy]; action = [ax, ay] (a push). Goal = the origin.

In [ ]:
DT, FR = 0.1, 0.1
goal = torch.tensor([0.0, 0.0], device=device)
def step_env(s, a):
    a = a.clamp(-1, 1)
    x, y, vx, vy = s[..., 0], s[..., 1], s[..., 2], s[..., 3]
    vx = vx * (1 - FR) + a[..., 0] * DT; vy = vy * (1 - FR) + a[..., 1] * DT
    return torch.stack([x + vx * DT, y + vy * DT, vx, vy], -1)
def reward(s): return -torch.linalg.norm(s[..., :2] - goal, dim=-1)

## 2 · Collect random experience

In [ ]:
s = torch.cat([torch.rand(3000, 2, device=device) * 4 - 2, torch.zeros(3000, 2, device=device)], -1)
S, A, S2 = [], [], []
for _ in range(20):
    a = torch.rand(s.shape[0], 2, device=device) * 2 - 1
    s2 = step_env(s, a); S.append(s); A.append(a); S2.append(s2); s = s2
S, A, S2 = torch.cat(S), torch.cat(A), torch.cat(S2)
print("transitions:", S.shape[0])

## 3 · Train the dynamics model  f(s,a) → Δs

In [ ]:
class Dyn(nn.Module):
    def __init__(self, H=256):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(6, H), nn.SiLU(), nn.Linear(H, H), nn.SiLU(), nn.Linear(H, 4))
    def forward(self, s, a): return self.net(torch.cat([s, a], -1))
model = Dyn().to(device); opt = torch.optim.Adam(model.parameters(), 1e-3)
inp = torch.cat([S, A], -1); tgt = S2 - S; hist = []
for step in range(STEPS + 1):
    idx = torch.randint(0, S.shape[0], (512,), device=device)
    loss = ((model(S[idx], A[idx]) - tgt[idx]) ** 2).mean()
    opt.zero_grad(); loss.backward(); opt.step()
    if step % max(1, STEPS // 10) == 0:
        hist.append((step, loss.item())); print(f"step {step:5d}  dyn MSE {loss.item():.5f}")

## 4 · Plan inside the model (random-shooting MPC)
At each step, imagine many random action sequences *with the learned model*, keep the first action of the best one, execute it in the real environment, repeat.

In [ ]:
@torch.no_grad()
def plan(s0, horizon=15, K=400, iters=4, elite=40):
    # Cross-Entropy Method: refit a Gaussian over action sequences toward the elites
    mu = torch.zeros(horizon, 2, device=device); std = torch.ones(horizon, 2, device=device)
    for _ in range(iters):
        acts = (mu + std * torch.randn(K, horizon, 2, device=device)).clamp(-1, 1)
        s = s0.repeat(K, 1); total = torch.zeros(K, device=device)
        for h in range(horizon):
            s = s + model(s, acts[:, h]); total = total + reward(s)
        idx = total.topk(elite).indices
        mu = acts[idx].mean(0); std = acts[idx].std(0) + 1e-3
    return mu[0]

def run(policy, n=40):
    s = torch.tensor([1.8, -1.6, 0.0, 0.0], device=device); traj = [s[:2].clone()]
    for _ in range(n):
        s = step_env(s, policy(s)); traj.append(s[:2].clone())
    return torch.stack(traj).cpu()
traj_plan = run(lambda s: plan(s))
traj_rand = run(lambda s: torch.rand(2, device=device) * 2 - 1)
print(f"final distance to goal — planner {traj_plan[-1].norm():.3f}  vs random {traj_rand[-1].norm():.3f}")

## 5 · Compare — planning in imagination reaches the goal

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].plot(traj_rand[:, 0], traj_rand[:, 1], "C7-o", ms=3, label="random")
ax[0].plot(traj_plan[:, 0], traj_plan[:, 1], "C0-o", ms=3, label="planned")
ax[0].scatter([0], [0], c="C3", s=80, marker="*", label="goal"); ax[0].legend(); ax[0].set_aspect("equal"); ax[0].set_title("trajectories")
ax[1].plot(*zip(*hist), "-o"); ax[1].set_title("dynamics MSE ↓"); ax[1].set_xlabel("step"); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

## Save artifacts — checkpoint · metrics · figure
Everything is written to **outputs/D_world_model/** — the model checkpoint, the full loss/eval history as JSON, and the final figure — then zipped. Colab sessions are ephemeral, so the last lines show how to download the zip or copy it to Google Drive.

In [ ]:
import os, json, torch, shutil
run = "outputs/D_world_model"; os.makedirs(run, exist_ok=True)
torch.save(model.state_dict(), f"{run}/dynamics.pt")
json.dump({"dyn_mse": hist, "final_dist": {"planner": float(traj_plan[-1].norm()), "random": float(traj_rand[-1].norm())}}, open(f"{run}/metrics.json", "w"), indent=2)
try:
    fig.savefig(f"{run}/figure.png", dpi=120, bbox_inches="tight")
except Exception: pass
shutil.make_archive(run, "zip", run)
print("saved to", run, "->", sorted(os.listdir(run)))
# keep it past the session:  from google.colab import files; files.download(f"{run}.zip")
# or mount Drive:  from google.colab import drive; drive.mount('/content/drive')  # then shutil.copytree(run, "/content/drive/MyDrive/"+run)

## (Optional) Publish to the Hugging Face Hub
Upload this run as a **model repo** — the checkpoint, `metrics.json` (full loss/eval history) and the results figure, embedded in an auto-generated model card. Do it for each lab, then group them into a **Collection** on your HF profile (Profile → New collection), or with the commented `add_collection_item` call below. It needs a **write token**, so it only runs once you sign in.

In [ ]:
# (optional) publish this run as a Hugging Face model repo — checkpoint + metrics + plot
!pip -q install huggingface_hub
from huggingface_hub import HfApi, notebook_login
import os
notebook_login()   # paste a WRITE token from https://huggingface.co/settings/tokens
api = HfApi(); user = api.whoami()["name"]
lab = os.path.basename(run); repo_id = f"{user}/ropedia-" + lab.lower().replace("_", "-")
fig = "\n![results](figure.png)\n" if os.path.exists(f"{run}/figure.png") else ""
open(f"{run}/README.md", "w").write("---\ntags: [ropedia-academy, education]\n---\n# " + lab + "\n\nTrained in **Ropedia Academy** (educational lab). Checkpoint, full loss/eval history (metrics.json) and the results figure are included." + fig)
api.create_repo(repo_id, repo_type="model", exist_ok=True)
api.upload_folder(folder_path=run, repo_id=repo_id, commit_message="Upload from Ropedia Academy")
print("uploaded ->", "https://huggingface.co/" + repo_id)
# group everything into one Collection (run once, after a few uploads):
# from huggingface_hub import create_collection, add_collection_item
# col = create_collection("Ropedia Academy - trained models", namespace=user, exists_ok=True)
# add_collection_item(col.slug, item_id=repo_id, item_type="model")

## How this links to tracks A–D
World models are the core of **AG · Agents & RL**.

### Where to go next
- Replace random shooting with **CEM** or learn a policy/value (actor-critic) inside the model → **Dreamer**.
- Learn the dynamics in a **latent** space from pixels (encoder + recurrent state) → the full world-model recipe.
- Swap the toy env for a Gym/MuJoCo task and the same loop scales up.